In [ ]:
from collections import deque
import spot


class LTLMutator:
    def __init__(self, ap_list):
        self.ap_set = set(ap_list)
        self.aps = [spot.formula.ap(p) for p in self.ap_set]
        
        self.unary_ops = {
            spot.op_Not: spot.formula.Not,
            spot.op_X: spot.formula.X,
            spot.op_F: spot.formula.F,
            spot.op_G: spot.formula.G
        }
        
        self.binary_ops = {
            spot.op_U: spot.formula.U,
            spot.op_W: spot.formula.W,
            spot.op_And: lambda a, b: spot.formula.And([a, b]),
            spot.op_Or: lambda a, b: spot.formula.Or([a, b])
        }
        
        self.rule3d_ops = {
            'U': spot.formula.U,
            'W': spot.formula.W,
            '&': lambda a, b: spot.formula.And([a, b]),
            '|': lambda a, b: spot.formula.Or([a, b])
        }

    def mutate(self, phi):
        if isinstance(phi, str):
            phi = spot.formula(phi)
            
        phi = phi.unabbreviate("ie")
        mutations = {}
        
        for mut_f in self._mutate_recursive(phi):
            s = mut_f.to_str()
            if s not in mutations:
                mutations[s] = mut_f
                
        orig_s = phi.to_str()
        if orig_s in mutations:
            del mutations[orig_s]
            
        return list(mutations.values())

    
                    
    def _mutate_recursive(self, f):
        # --- GENERAL CASES (Strictly restricted based on your structural goal) ---
        # Rule 5: Wrap the current sub-formula in a unary operator
        for op_func in self.unary_ops.values():
            yield op_func(f)
            
        kind = f.kind()
        
        # --- BASE CASES / LEAF MUTATIONS ---
        # Rule 1 & Rule 6 (Moved here so they only mutate leaves, not the whole tree)
        if f.is_tt():
            yield spot.formula.ff()
            
        if f.is_ff():
            yield spot.formula.tt()
            

        if f in self.ap_set:
            # Rule 2: Swap APs
            for p in self.aps:
                if p != f:
                    yield p
            # Rule 6 for APs: allow mapping an AP to other APs (handled above)
        

        # --- INDUCTIVE CASES ---
        if kind in self.unary_ops:
            child = f[0]
            
            # 3(a) Change unary operator
            for other_kind, op_func in self.unary_ops.items():
                if other_kind != kind:
                    yield op_func(child)
                    
            # 3(b) Drop operator
            yield child
            
            # 3(c) Mutate child 
            for mutated_child in self._mutate_recursive(child):
                yield self.unary_ops[kind](mutated_child)
                
            # 3(d) Append binary operator
            for p in self.aps:
                for op_func in self.rule3d_ops.values():
                    yield op_func(p, f)
                    
        elif kind in self.binary_ops:
            children = list(f)
            
            if len(children) >= 2:
                phi_1 = children[0]
                phi_2 = children[1]
                if len(children) > 2:
                    if kind == spot.op_And:
                        phi_2 = spot.formula.And(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    elif kind == spot.op_Or:
                        phi_2 = spot.formula.Or(children[1:])
                    
                # 4(a) Change binary operator
                for other_kind, op_func in self.binary_ops.items():
                    if other_kind != kind:
                        yield op_func(phi_1, phi_2)
                        
                # 4(b) Keep one child
                yield phi_1
                yield phi_2
                
                # 4(c) Mutate left child
                for mutated_left in self._mutate_recursive(phi_1):
                    yield self.binary_ops[kind](mutated_left, phi_2)
                    
                # 4(d) Mutate right child
                for mutated_right in self._mutate_recursive(phi_2):
                    yield self.binary_ops[kind](phi_1, mutated_right)


In [23]:
# 1. Define the pool of atomic propositions used in your system
ap_pool = ['p', 'q']

# 2. Instantiate the Mutator
mutator = LTLMutator(ap_pool)

# 3. Supply the base LTL formula you wish to mutate
original_formula = "G(p U q)"
mutations = mutator.mutate(original_formula)

print(f"Original Formula: {original_formula}")
print(f"Total Unique Mutations Found: {len(mutations)}")
print("-" * 30)

# 4. Display the results
for i, m in enumerate(mutations):
    print(f"{i + 1:02d}: {m.to_str()}")

Original Formula: G(p U q)
Total Unique Mutations Found: 31
------------------------------
01: !G(p U q)
02: XG(p U q)
03: FG(p U q)
04: !(p U q)
05: X(p U q)
06: F(p U q)
07: p U q
08: G!(p U q)
09: GX(p U q)
10: GF(p U q)
11: G(p W q)
12: G(p & q)
13: G(p | q)
14: Gp
15: Gq
16: G(!p U q)
17: G(Xp U q)
18: G(Fp U q)
19: G(Gp U q)
20: G(p U !q)
21: G(p U Xq)
22: G(p U Fq)
23: G(p U Gq)
24: q U G(p U q)
25: q W G(p U q)
26: q & G(p U q)
27: q | G(p U q)
28: p U G(p U q)
29: p W G(p U q)
30: p & G(p U q)
31: p | G(p U q)


In [ ]:
# swap unary operators!!!!!

7

In [28]:
spot.formula.And(spot.formula("G(p U q)"))

TypeError: in method 'formula_And', argument 1 of type 'std::vector< spot::formula,std::allocator< spot::formula > > const &'
Additional information:
Wrong number or type of arguments for overloaded function 'formula_And'.
  Possible C/C++ prototypes are:
    spot::formula::And(std::vector< spot::formula,std::allocator< spot::formula > > const &)
    spot::formula::And(spot::formula const &,spot::formula const &)
